# Post-Experiment Analysis & High-Res Visualizations

This notebook takes the data already extracted from the identification and ablation passes and generates large, high-DPI, presentation-ready visualizations. You can tweak the styling (colors, font sizes) here without re-running any heavy LLM inferences.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

### 1. Configuration & Constants
Point `BASE_DIR` to the specific run directory you want to analyze.

In [ ]:
# --- CONFIGURATION ---
# Change this if running in Colab to match your mount path (e.g., "/content/data/...")
# Assuming you are running this from the repository root or similar.
RUN_NAME = "llama3-8B_instruct_run_2026-03-07_23-38-50"

BASE_DIR = f"../../../../data/retrieval_heads/{RUN_NAME}" 
# If running in Colab, you might need: BASE_DIR = f"/content/data/retrieval_heads/{RUN_NAME}"

RAW_OUTPUT_DIR = os.path.join(BASE_DIR, "raw")
RESULTS_OUTPUT_DIR = os.path.join(BASE_DIR, "results")
ABLATION_OUTPUT_DIR = os.path.join(BASE_DIR, "ablation_results")
ANALYSIS_OUTPUT_DIR = os.path.join(BASE_DIR, "analysis_plots")

os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)
print(f"High-res analysis plots will be saved to: {ANALYSIS_OUTPUT_DIR}")

---
## 2. Identification Graphs (High Resolution)

In [ ]:
# Load Identification Data
heads_json_path = os.path.join(RESULTS_OUTPUT_DIR, "retrieval_heads.json")
try:
    with open(heads_json_path, "r") as f:
        heads_data = json.load(f)
    
    task_top_heads = heads_data["tasks"]
    shared_heads = heads_data["shared_heads"]
    
    # Load raw NumPy arrays to recreate the heatmaps
    task_mean_scores = {}
    for task_id in task_top_heads.keys():
        files = [f for f in os.listdir(RAW_OUTPUT_DIR) if f.endswith(f"_{task_id}.npy")]
        if files:
            stacked = np.stack([np.load(os.path.join(RAW_OUTPUT_DIR, f)) for f in files])
            task_mean_scores[task_id] = stacked.mean(axis=0)
            
    print(f"Loaded {len(task_mean_scores)} task matrices.")
    print(f"Loaded {len(shared_heads)} shared heads.")
    
except FileNotFoundError:
    print(f"Could not find {heads_json_path}. Make sure the paths are correct.")

In [ ]:
# Enlarged Per-Task Heatmaps
for task_id, mean_scores in task_mean_scores.items():
    fig, ax = plt.subplots(figsize=(16, 16), facecolor='white')
    im = ax.imshow(mean_scores, aspect="equal", cmap="viridis")

    ax.set_title(f"Sum Attention Scores — {task_id}", fontsize=24, fontweight='bold', pad=20)
    ax.set_xlabel("Head Index", fontsize=18, fontweight='bold', labelpad=10)
    ax.set_ylabel("Layer Index", fontsize=18, fontweight='bold', labelpad=10)
    
    ax.set_xticks(np.arange(mean_scores.shape[1]))
    ax.set_yticks(np.arange(mean_scores.shape[0]))
    ax.tick_params(axis='both', which='major', labelsize=10)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Mean Sum Attention", size=18, weight='bold')
    cbar.ax.tick_params(labelsize=14)

    # Red plus markers for the top heads
    for h in task_top_heads[task_id]:
        ax.plot(h["head"], h["layer"], "r+", markersize=14, markeredgewidth=3)

    plt.tight_layout()
    fig_path = os.path.join(ANALYSIS_OUTPUT_DIR, f"heatmap_{task_id}_large.png")
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close() # Close to prevent huge inline displays if running locally
    print(f"Saved: {fig_path}")

In [ ]:
# Enlarged Shared Heads Grid
if shared_heads:
    sample_shape = next(iter(task_mean_scores.values())).shape
    num_layers, num_heads = sample_shape
    grid = np.zeros((num_layers, num_heads))
    
    for h in shared_heads:
        grid[h["layer"], h["head"]] = h["avg_score"]

    fig, ax = plt.subplots(figsize=(16, 16), facecolor='white')
    im = ax.imshow(grid, aspect="equal", cmap="hot")
    
    ax.set_title("Shared Retrieval Heads — All Tasks", fontsize=24, fontweight='bold', pad=20)
    ax.set_xlabel("Head Index", fontsize=18, fontweight='bold', labelpad=10)
    ax.set_ylabel("Layer Index", fontsize=18, fontweight='bold', labelpad=10)
    
    ax.set_xticks(np.arange(num_heads))
    ax.set_yticks(np.arange(num_layers))
    ax.tick_params(axis='both', which='major', labelsize=10)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Avg Sum Attention Score", size=18, weight='bold')
    cbar.ax.tick_params(labelsize=14)

    plt.tight_layout()
    fig_path = os.path.join(ANALYSIS_OUTPUT_DIR, "shared_retrieval_heads_large.png")
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fig_path}")

In [ ]:
# Enlarged Task Overlap Heatmap
task_ids = list(task_top_heads.keys())
n_tasks = len(task_ids)
overlap_matrix = np.zeros((n_tasks, n_tasks))

for i, t1 in enumerate(task_ids):
    set1 = set((h["layer"], h["head"]) for h in task_top_heads[t1])
    for j, t2 in enumerate(task_ids):
        set2 = set((h["layer"], h["head"]) for h in task_top_heads[t2])
        union = len(set1 | set2)
        overlap_matrix[i, j] = (len(set1 & set2) / union * 100) if union > 0 else 0

clean_names = [t.replace('_', ' ').title().replace(' Count', '') for t in task_ids]

fig, ax = plt.subplots(figsize=(12, 10), facecolor='white')
cax = ax.imshow(overlap_matrix, cmap='Blues')

cbar = plt.colorbar(cax)
cbar.set_label('Overlap (%)', size=16, weight='bold')
cbar.ax.tick_params(labelsize=12)

ax.set_xticks(np.arange(n_tasks))
ax.set_yticks(np.arange(n_tasks))
ax.set_xticklabels(clean_names, rotation=45, ha='right', fontsize=14, weight='bold')
ax.set_yticklabels(clean_names, fontsize=14, weight='bold')

for i in range(n_tasks):
    for j in range(n_tasks):
        val = overlap_matrix[i, j]
        color = 'white' if val > 50 else 'black'
        text = f"{val:.0f}%" if i != j else "-"
        ax.text(j, i, text, ha='center', va='center', color=color, fontsize=14, weight='bold')

plt.title("Head Overlap Between Tasks", fontsize=22, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = os.path.join(ANALYSIS_OUTPUT_DIR, "task_overlap_heatmap_large.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

### Space for New/Custom Identification Graphs
Add any new graphs relying solely on the Identification data here.

In [ ]:
# [Add custom identification visualizations here]



---
## 3. Ablation Graphs (High Resolution)
Ensure you have run the `retrieval_head_ablation.ipynb` notebook and saved the JSON files first.

In [ ]:
# Load Ablation Data
try:
    with open(os.path.join(ABLATION_OUTPUT_DIR, "baseline_results.json"), "r") as f:
        baseline_results = json.load(f)

    with open(os.path.join(ABLATION_OUTPUT_DIR, "within_task_ablation.json"), "r") as f:
        within_task_results = json.load(f)

    with open(os.path.join(ABLATION_OUTPUT_DIR, "across_task_ablation.json"), "r") as f:
        across_task_results = json.load(f)
        
    # Random ablation might not exist if it wasn't run yet, so handle gracefully
    random_ablation_path = os.path.join(ABLATION_OUTPUT_DIR, "random_ablation_control.json")
    random_ablation_results = {}
    if os.path.exists(random_ablation_path):
        with open(random_ablation_path, "r") as f:
            random_ablation_results = json.load(f)

    baseline_overall = sum(r["matches"] for r in baseline_results.values()) / max(1, sum(r["attempts"] for r in baseline_results.values()))
    print(f"Loaded ablation data. Baseline overall: {baseline_overall:.1%}")
except FileNotFoundError as e:
    print(f"Error loading ablation data: {e}")
    print("Make sure you have run the ablation experiments and saved the JSON results.")

In [ ]:
# Enlarged Across-task vs Random Ablation
fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')

# Ensure integer keys for sorting
k_vals = sorted([int(k) for k in across_task_results.keys()])
retrieval_acc = [across_task_results[str(k)]["overall_accuracy"] for k in k_vals]

random_acc = []
if random_ablation_results:
    random_acc = [random_ablation_results[str(k)]["overall_accuracy"] for k in k_vals if str(k) in random_ablation_results]

ax.axhline(y=baseline_overall, color="green", linestyle="--", linewidth=3, label=f"Baseline ({baseline_overall:.1%})")
ax.plot(k_vals, retrieval_acc, "ro-", linewidth=4, markersize=12, label="Retrieval Heads Ablated")

if random_acc:
    ax.plot(k_vals[:len(random_acc)], random_acc, "bs-", linewidth=4, markersize=12, label="Random Heads Ablated")

ax.set_xlabel("Number of Heads Ablated (K)", fontsize=18, fontweight='bold', labelpad=15)
ax.set_ylabel("Overall Accuracy", fontsize=18, fontweight='bold', labelpad=15)
ax.set_title("Shared Retrieval Head Ablation vs Random Control", fontsize=24, fontweight='bold', pad=20)

ax.legend(fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=16)
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
fig_path = os.path.join(ANALYSIS_OUTPUT_DIR, "across_task_vs_random_large.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# Enlarged Within-task Ablation per Task
fig, ax = plt.subplots(figsize=(16, 9), facecolor='white')

for task_id, k_results in within_task_results.items():
    k_vals = sorted([int(k) for k in k_results.keys()])
    acc_vals = [k_results[str(k)]["accuracy"] for k in k_vals]
    
    display_name = clean_names[task_ids.index(task_id)] if task_id in task_ids else task_id
    ax.plot(k_vals, acc_vals, "o-", linewidth=3, markersize=10, label=display_name)

ax.axhline(y=baseline_overall, color="black", linestyle="--", linewidth=4, label="Overall Baseline")

ax.set_xlabel("Number of Heads Ablated (K)", fontsize=18, fontweight='bold', labelpad=15)
ax.set_ylabel("Task Accuracy", fontsize=18, fontweight='bold', labelpad=15)
ax.set_title("Within-Task Ablation: Accuracy Drop by Task", fontsize=24, fontweight='bold', pad=20)

ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=16)
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
fig_path = os.path.join(ANALYSIS_OUTPUT_DIR, "within_task_ablation_large.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# Enlarged Accuracy Drop Bar Chart
max_k = max([int(k) for k in next(iter(within_task_results.values())).keys()])
task_drops = []
task_labels = []

for task_id, results in within_task_results.items():
    if str(max_k) in results:
        base_acc = baseline_results[task_id]['accuracy']
        ablated_acc = results[str(max_k)]['accuracy']
        task_drops.append((base_acc - ablated_acc) * 100)
        display_name = clean_names[task_ids.index(task_id)] if task_id in task_ids else task_id
        task_labels.append(display_name)

# Sort by largest drop
sorted_idx = np.argsort(task_drops)[::-1]
task_drops = np.array(task_drops)[sorted_idx]
task_labels = np.array(task_labels)[sorted_idx]

plt.figure(figsize=(14, 8), facecolor='white')
bars = plt.bar(task_labels, task_drops, color='steelblue', edgecolor='black', linewidth=1.5)

plt.ylabel("Accuracy Drop (%)", fontsize=18, fontweight='bold', labelpad=15)
plt.title(f"Impact of Ablating Top {max_k} Heads", fontsize=24, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=16, fontweight='bold')
plt.yticks(fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f'{yval:.1f}%', 
             ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
fig_path = os.path.join(ANALYSIS_OUTPUT_DIR, "accuracy_drop_summary_large.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

### Space for New/Custom Ablation Graphs
Add any new graphs relying solely on the Ablation data here.

In [ ]:
# [Add custom ablation visualizations here]

